# 4. Score, compare and tune

Every method goes through the same steps: cleanup, evaluation against reference outlines, a per-building comparison, tuning on one area, freezing the settings, and one score on a separate test area. This notebook runs all of them on the synthetic demo in seconds, with two prediction sets:

- **identity:** exact copies of the reference outlines, so every metric should be perfect;
- **esri_format:** the demo's Esri-layout file, with a duplicate, a self-intersecting outline, a missed building and a false detection.

In [1]:
import json
import shutil
from pathlib import Path

from nearmap_buildings.cli import main as _nbf

# Run from the repository root or from notebooks/. Generated files go to notebook_runs/, which git ignores.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORK = REPO / "notebook_runs" / "04_score_and_compare"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)


def nbf(*args):
    """Run one nbf command in this process; stop the notebook if it fails."""
    code = _nbf([str(a) for a in args])
    if code:
        raise RuntimeError(f"nbf {args[0]} failed with exit code {code}")

In [2]:
import geopandas as gpd
import pandas as pd

METRIC = "EPSG:32614"
DEMO = WORK / "demo"
nbf("demo", "--output", DEMO)
sorted(p.name for p in DEMO.iterdir())

<repo>\notebook_runs\04_score_and_compare\demo


['NOTICE.json',
 'aoi.geojson',
 'boxes.geojson',
 'esri_format_predictions.gdb',
 'exemplars.geojson',
 'imagery.tif',
 'perfect_predictions.gpkg',
 'points.geojson',
 'reference.gpkg',
 'splits.geojson']

## Clean

The same cleanup for every method: repair, drop pieces under 4 m², and suppress duplicates at IoU 0.7 or 98% containment, keeping the complete or higher-scoring copy.

In [3]:
nbf("clean", "--input", DEMO / "perfect_predictions.gpkg", "--output", WORK / "identity.gpkg", "--metric-crs", METRIC)
nbf("import-vectors", "--input", DEMO / "esri_format_predictions.gdb", "--layer", "esri_buildings", "--preset", "esri",
    "--crs", METRIC, "--output", WORK / "esri_raw.gpkg")
nbf("clean", "--input", WORK / "esri_raw.gpkg", "--output", WORK / "esri_format.gpkg", "--metric-crs", METRIC)

{"input_count": 9, "cleaned_count": 9, "removed_count": 0, "duplicate_priority": "complete", "raw_inputs_preserved": true}
{
  "output": "<repo>\\notebook_runs\\04_score_and_compare\\esri_raw.gpkg",
  "features": 10,
  "crs": "EPSG:32614",
  "method": "esri_building_usa",
  "source_id": "source feature ID",
  "score_field": "Confidence",
  "score_scale": 100.0,
  "invalid_polygons_kept": 1
}


{"input_count": 10, "cleaned_count": 9, "removed_count": 1, "duplicate_priority": "complete", "raw_inputs_preserved": true}


## Evaluate

Matching is one-to-one at IoU ≥ 0.5, in a metric CRS, inside an area of interest; objects crossing its edge are excluded. Each report must say what the reference is:

- `--independent-holdout`: reference outlines drawn independently for the imagery date. The demo's are exact by construction.
- `--agreement-only`: an existing inventory that may be outdated. Lewisville's 2015 outlines are older than the 2026 imagery, so every Lewisville score is agreement, not accuracy.

In [4]:
reports = {}
for name in ("identity", "esri_format"):
    (WORK / name).mkdir()
    nbf("evaluate", "--predictions", WORK / f"{name}.gpkg", "--predictions-layer", "cleaned",
        "--reference", DEMO / "reference.gpkg", "--aoi", DEMO / "aoi.geojson", "--metric-crs", METRIC,
        "--output-json", WORK / name / "evaluation.json", "--output-gpkg", WORK / name / "evaluation.gpkg", "--independent-holdout")
    reports[name] = json.loads((WORK / name / "evaluation.json").read_text())

pd.DataFrame({name: {k: r[k] for k in ("true_positives", "false_positives", "false_negatives", "precision", "recall", "f1")}
              for name, r in reports.items()}).T.round(3)

{"true_positives": 9, "false_positives": 0, "false_negatives": 0, "precision": 1.0, "recall": 1.0, "f1": 1.0}


{"true_positives": 8, "false_positives": 1, "false_negatives": 1, "precision": 0.8888888888888888, "recall": 0.8888888888888888, "f1": 0.8888888888888888}


,true_positives,false_positives,false_negatives,precision,recall,f1
identity,9.0,0.0,0.0,1.000,1.000,1.000
esri_format,8.0,1.0,1.0,0.889,0.889,0.889


## Compare per building

`nbf compare` checks that the reports share a reference and settings, writes a table, and with `--per-building` records which methods found each reference building.

In [5]:
nbf("compare", "--reports", WORK / "identity" / "evaluation.json", WORK / "esri_format" / "evaluation.json",
    "--labels", "identity", "esri_format", "--output", WORK / "comparison.csv", "--per-building", WORK / "per_building.gpkg")
per_building = gpd.read_file(WORK / "per_building.gpkg", layer="buildings")
per_building.outcome.value_counts()

{"reports": 2, "output": "<repo>\\notebook_runs\\04_score_and_compare\\comparison.csv", "same_holdout_and_settings": true, "undefined_metrics": "empty CSV cells", "per_building": "<repo>\\notebook_runs\\04_score_and_compare\\per_building.gpkg", "outcomes": {"found by both": 8, "only identity": 1}}


outcome
found by both    8
only identity    1
Name: count, dtype: int64

## Tune on one area, freeze, then score the test area once

The Lewisville protocol, in miniature:

1. Fix the objective before looking at results. Lewisville used F1 on main buildings of 20 m² and up.
2. Score every variant on the tuning area.
3. Take the best F1. Among variants within 0.01 of it, keep the one closest to the defaults.
4. Freeze those settings, then score the test area exactly once.

The demo's validation region serves as the tuning area and its test region as the test area. The variants are cut-offs for the Esri-format scores, with regularization off and on.

In [6]:
from nearmap_buildings.evaluation import evaluate
from nearmap_buildings.postprocess import clean_polygons

splits = gpd.read_file(DEMO / "splits.geojson")
tuning_area, test_area = splits[splits.split == "val"], splits[splits.split == "test"]
reference = gpd.read_file(DEMO / "reference.gpkg")
raw = gpd.read_file(WORK / "esri_raw.gpkg")


def score(threshold, regularize, area):
    kept = raw[raw.score >= threshold].reset_index(drop=True)
    options = {"regularize": True, "max_displacement": 0.5, "max_area_change": 0.10} if regularize else {}
    cleaned = clean_polygons(kept, crs=METRIC, min_area=4, **options).cleaned
    report = evaluate(cleaned, reference, area, crs=METRIC, reference_status="independent_holdout").report
    return {k: report[k] for k in ("true_positives", "false_positives", "false_negatives", "precision", "recall", "f1")}


grid = pd.DataFrame([{"threshold": t, "regularize": r, **score(t, r, tuning_area)}
                     for t in (0.5, 0.6, 0.7, 0.8, 0.9, 0.95) for r in (False, True)])
grid.round(3)

,threshold,regularize,true_positives,false_positives,false_negatives,precision,recall,f1
0,0.50,False,2,0,1,1.0,0.667,0.8
1,0.50,True,2,0,1,1.0,0.667,0.8
2,0.60,False,2,0,1,1.0,0.667,0.8
3,0.60,True,2,0,1,1.0,0.667,0.8
4,0.70,False,2,0,1,1.0,0.667,0.8
5,0.70,True,2,0,1,1.0,0.667,0.8
6,0.80,False,2,0,1,1.0,0.667,0.8
7,0.80,True,2,0,1,1.0,0.667,0.8
8,0.90,False,2,0,1,1.0,0.667,0.8
9,0.90,True,2,0,1,1.0,0.667,0.8


In [7]:
DEFAULTS = {"threshold": 0.5, "regularize": False}


def closest_to_defaults(row):
    changed = (row.threshold != DEFAULTS["threshold"]) + (row.regularize != DEFAULTS["regularize"])
    return (changed, abs(row.threshold - DEFAULTS["threshold"]))


best = grid.f1.max()
frozen = min(grid[grid.f1 >= best - 0.01].itertuples(), key=closest_to_defaults)
FROZEN = {"threshold": frozen.threshold, "regularize": bool(frozen.regularize)}
(WORK / "frozen_settings.json").write_text(json.dumps(FROZEN))
print("best F1 on the tuning area:", round(best, 3), "| frozen:", FROZEN)

best F1 on the tuning area: 0.8 | frozen: {'threshold': 0.5, 'regularize': False}


In [8]:
def score_test_area_once(settings, record=WORK / "test_area_scores.json"):
    """Score the frozen settings on the test area; refuse to score it a second time."""
    if record.exists():
        raise RuntimeError("The test area has already been scored.")
    result = score(settings["threshold"], settings["regularize"], test_area)
    record.write_text(json.dumps(result))
    return result


score_test_area_once(FROZEN)

{'true_positives': 3,
 'false_positives': 1,
 'false_negatives': 0,
 'precision': 0.75,
 'recall': 1.0,
 'f1': 0.8571428571428571}

## Lewisville

The same steps, run on the city-wide outputs of notebooks 2 and 3 and the city's published Esri run, gave these final results. They are agreement with the 2015 outlines, not accuracy.

| Main buildings of 20 m² and up | Esri, city run | Esri, local raw run | SAM 3 |
|---|---|---|---|
| City-wide F1 | 0.83 | 0.77 | 0.74 |
| Test area F1, scored once | 0.95 | 0.91 | 0.87 |

Full tables, charts and the tuning protocol: [results/lewisville](../results/lewisville/README.md).